# InsurAI — ML Model Training
This notebook trains an XGBoost risk prediction model on the Kaggle Car Insurance dataset.
At the end it saves `backend/models/risk_model.pkl` for the Flask API to use.

**Dataset:** https://www.kaggle.com/datasets/sagnik1511/car-insurance-data  
Download `Car_Insurance_Claim.csv` and place it in this `notebooks/` folder.

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# !pip install pandas scikit-learn xgboost joblib matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

print('Libraries loaded ✓')

## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv('Car_Insurance_Claim.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('\nMissing values:')
print(df.isnull().sum())
print('\nTarget distribution:')
print(df['OUTCOME'].value_counts())

## 2. Clean & Engineer Features

In [ ]:
# Drop duplicates
df.drop_duplicates(inplace=True)

# Standardise column names to lowercase
df.columns = df.columns.str.lower().str.strip()

# Map target: 1 = filed a claim (risky), 0 = no claim (safe)
# Adjust column name to match the actual dataset
target_col = 'outcome'  # change if different in your CSV
df['target'] = df[target_col].astype(int)

# Select feature columns — adjust names to match actual dataset columns
FEATURE_COLS = [
    'age', 'gender', 'race', 'driving_experience', 'education',
    'income', 'credit_score', 'vehicle_ownership', 'vehicle_year',
    'married', 'children', 'postal_code', 'annual_mileage',
    'vehicle_type', 'speeding_violations', 'duis', 'past_accidents'
]

# Keep only columns that exist in the dataset
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
print(f'Using {len(FEATURE_COLS)} features: {FEATURE_COLS}')

X = df[FEATURE_COLS].copy()
y = df['target']

In [ ]:
# ── Encode categoricals ───────────────────────────────────────────────────────
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
print(f'Encoding: {categorical_cols}')

le_dict = {}
for col in categorical_cols:
    le = LabelEncoder()
    # Handle NaN by filling with 'unknown' before encoding
    X[col] = X[col].fillna('unknown')
    X[col] = le.fit_transform(X[col].astype(str))
    le_dict[col] = le

# Fill remaining numeric NaN with median
numeric_cols = X.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    X[col].fillna(X[col].median(), inplace=True)

print('Encoding complete ✓')
X.head()

In [ ]:
# ── Engineered features ───────────────────────────────────────────────────────
if 'past_accidents' in X.columns and 'age' in X.columns:
    X['accident_rate'] = X['past_accidents'] / (X['age'] - 17).clip(lower=1)

if 'annual_mileage' in X.columns and 'vehicle_year' in X.columns:
    vehicle_age_proxy = 2024 - X['vehicle_year'].clip(upper=2024)
    X['mileage_per_vehicle_year'] = X['annual_mileage'] / vehicle_age_proxy.clip(lower=1)

if 'age' in X.columns:
    X['young_driver']  = (X['age'] < 25).astype(int)
    X['senior_driver'] = (X['age'] > 70).astype(int)

print('Feature engineering complete ✓')
print(f'Total features: {X.shape[1]}')

## 3. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Positive rate — train: {y_train.mean():.2%}  test: {y_test.mean():.2%}')

## 4. Train Models & Compare

In [ ]:
# ── Random Forest ─────────────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=10,
    class_weight='balanced',  # handles class imbalance
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
print(f'Random Forest AUC: {rf_auc:.4f}')

In [ ]:
# ── XGBoost ───────────────────────────────────────────────────────────────────
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    use_label_encoder=False,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)
xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

xgb_auc = roc_auc_score(y_test, xgb.predict_proba(X_test)[:, 1])
print(f'\nXGBoost AUC: {xgb_auc:.4f}')

In [ ]:
# Pick the better model
best_model = xgb if xgb_auc >= rf_auc else rf
best_name  = 'XGBoost' if xgb_auc >= rf_auc else 'RandomForest'
print(f'Best model: {best_name} (AUC={max(xgb_auc, rf_auc):.4f})')

# Full classification report
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Safe','Risky']))

## 5. Visualise Feature Importance

In [ ]:
importances = best_model.feature_importances_
feat_df = pd.DataFrame({'feature': X.columns, 'importance': importances})
feat_df = feat_df.sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_df, x='importance', y='feature', palette='viridis')
plt.title(f'{best_name} — Top 15 Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()
print('Saved feature_importance.png')

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Safe','Risky'], yticklabels=['Safe','Risky'])
plt.title('Confusion Matrix')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## 6. Save Model

In [ ]:
output_dir = '../backend/models'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'risk_model.pkl')

joblib.dump(best_model, output_path)
print(f'Model saved to {output_path} ✓')

# Also save feature list for the preprocessing module
feature_path = os.path.join(output_dir, 'feature_columns.txt')
with open(feature_path, 'w') as f:
    f.write('\n'.join(X.columns.tolist()))
print(f'Feature list saved to {feature_path} ✓')

# Quick sanity check
loaded = joblib.load(output_path)
test_proba = loaded.predict_proba(X_test[:3])
print(f'\nSanity check — 3 predictions: {test_proba[:, 1].round(3)}')

## 7. Answers to Challenge Questions

### Q1 — How does the model decide high-risk vs low-risk?
The XGBoost model scores each application by combining hundreds of decision trees, each
trained on a random subset of features. The final risk score is the average predicted
probability across all trees. Features like `prior_claims`, `speeding_violations`, and
`age` have the highest importances (see chart above).  
Thresholds: score < 0.35 → low, 0.35–0.65 → medium, > 0.65 → high.

### Q2 — Can the AI handle missing data?
Yes. The preprocessing pipeline fills missing numeric values with the training-set median
and missing categoricals with the most common value ('unknown'). A user who skips
optional fields (e.g. `occupation`) still receives a valid quote — the model degrades
gracefully rather than throwing an error.

### Q3 — Biggest limitation
The model learns patterns from historical claims data, which may encode systemic biases
(e.g. geographic redlining, income proxies). Without an explicit fairness audit (e.g.
checking equal false-positive rates across demographic groups), the system could
perpetuate discriminatory pricing. Regulatory frameworks in many jurisdictions
(EU AI Act, US CFPB guidance) require explainability and bias testing before deployment.